In [3]:
import pandas as pd
from tardis import run_tardis
from tardis.io.configuration.config_reader import Configuration

In [4]:
print("Setting up TARDIS Configuration with Tracking...")
config = Configuration.from_yaml("tardis_example.yml")
config.montecarlo.tracking.track_rpacket = True

print("Running Simulation...")
sim = run_tardis(config, log_level="INFO", show_progress_bars=False)
print("Simulation Complete!")

Setting up TARDIS Configuration with Tracking...
Running Simulation...


Embedding the final state for Jupyter environments
Simulation Complete!


In [5]:
# Extracting the Tracker DataFrame
if hasattr(sim, 'transport_state') and getattr(sim.transport_state, 'tracker_full_df', None) is not None:
    df = sim.transport_state.tracker_full_df
elif hasattr(sim, 'transport') and getattr(sim.transport.transport_state, 'tracker_full_df', None) is not None:
    df = sim.transport.transport_state.tracker_full_df
elif hasattr(sim.transport, 'tracker_full_df') and getattr(sim.transport, 'tracker_full_df', None) is not None:
    df = sim.transport.tracker_full_df
else:
    raise ValueError("DataFrame is still None. Tracking failed to initialize.")

print("Total rows:", len(df))
df.head()

Total rows: 2085687


interaction_type      status        radius  \
packet_id event_id                                              
0         0                BOUNDARY  IN_PROCESS  1.235520e+15   
          1                    LINE  IN_PROCESS  1.277874e+15   
          2                BOUNDARY  IN_PROCESS  1.235520e+15   
          3                BOUNDARY  REABSORBED  1.235520e+15   
1         0                BOUNDARY  IN_PROCESS  1.235520e+15   

                    before_shell_id  after_shell_id     before_nu  before_mu  \
packet_id event_id                                                             
0         0                      -1               0  1.737373e+15   0.839765   
          1                       0               0  1.737373e+15   0.851130   
          2                       0              -1  6.403468e+14  -0.571333   
          3                       0               1  6.403468e+14  -0.571333   
1         0                      -1               0  3.867729e+14   0.777574   

                    before_energy      after_nu  after_mu  after_energy  \
packet_id event_id                                                        
0         0               0.00001  1.737373e+15  0.839765       0.00001   
          1               0.00001  6.403468e+14 -0.608549       0.00001   
          2               0.00001  6.403468e+14 -0.571333       0.00001   
          3               0.00001  6.403468e+14 -0.571333       0.00001   
1         0               0.00001  3.867729e+14  0.777574       0.00001   

                    line_absorb_id  line_emit_id  
packet_id event_id                                
0         0                     -1            -1  
          1                   5343         10663  
          2                     -1            -1  
          3                     -1            -1  
1         0                     -1            -1

In [6]:
# 1. Filter out 'BOUNDARY' interactions
df_filtered = df[df['interaction_type'] != 'BOUNDARY']

# 2. Find the most interesting packet
packet_counts = df_filtered.groupby(level='packet_id').size()
best_packet_id = packet_counts.idxmax()
num_interactions = packet_counts.max()

print(f"Selected Packet ID: {best_packet_id}")
print(f"Total non-boundary interactions: {num_interactions}")

# 3. Extract and display the history
packet_history = df_filtered.xs(best_packet_id, level='packet_id')
cols_to_show = ['interaction_type', 'status', 'radius', 'line_absorb_id', 'line_emit_id']
clean_history = packet_history[cols_to_show]

clean_history

Selected Packet ID: 38423
Total non-boundary interactions: 62


,interaction_type,status,radius,line_absorb_id,line_emit_id
event_id,,,,,
3,ESCATTERING,IN_PROCESS,1.358367e+15,-1,-1
6,ESCATTERING,IN_PROCESS,1.465547e+15,-1,-1
13,LINE,IN_PROCESS,1.741040e+15,10643,3519
14,LINE,IN_PROCESS,1.741333e+15,3520,3552
15,LINE,IN_PROCESS,1.754290e+15,3553,3551
...,...,...,...,...,...
116,LINE,IN_PROCESS,1.977324e+15,3543,3354
118,LINE,IN_PROCESS,1.997718e+15,3358,3332
119,LINE,IN_PROCESS,2.038574e+15,3341,3539
